<a href="https://www.kaggle.com/code/gpreda/convert-geojson-to-parquet-format?scriptVersionId=292889195" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# Introduction

This is a notebook to transform the format of the data in: [Bornes de Recharge pour Véhicules Électriques](https://www.kaggle.com/datasets/gpreda/bornes-de-recharge-pour-vhicules-lectriques) dataset.   

From the initial format of GeoJSON, we will transform the data in parquet.

Why this is important?

Parquet format will load much faster and you can apply additional filters on the Parquet file.



# Packages

In [1]:
!pip install --upgrade --quiet geopandas pyogrio shapely

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 341.7/341.7 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 32.5/32.5 MB 51.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 80.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.8.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
bigframes 2.8.0 requires google-cloud-bigquery[bqstorage,pandas]>=3.31.0, but you have google-cloud-bigquery 3.25.0 which is incompatible.
bigframes 2.8.0 requires rich<14,>=12.4.4, but you have rich 14.0.0 which is incompatible.


In [2]:
import geopandas as gpd
import pandas as pd
from shapely.geometry import Point

# Load the data


We load entire data, to save then as parquet file.

In [3]:
path = "/kaggle/input/bornes-de-recharge-pour-vhicules-lectriques/consolidation-etalab-schema-irve-statique-v-2.3.1-20250907.geojson"

# Fast read (falls back to fiona if pyogrio missing)
gdf = gpd.read_file(path, engine="pyogrio")

                    # Ensure CRS is WGS84 (GeoJSON usually is)
if gdf.crs is None:
    gdf = gdf.set_crs(4326)

Fallback if missing geometry.

In [4]:
missing_geom = gdf.geometry.isna()
if missing_geom.any() and {"consolidated_longitude","consolidated_latitude"}.issubset(gdf.columns):
    pts = gpd.points_from_xy(gdf.loc[missing_geom, "consolidated_longitude"],
                             gdf.loc[missing_geom, "consolidated_latitude"],
                             crs="EPSG:4326")
    gdf.loc[missing_geom, "geometry"] = pts

# Convert the data

In [5]:
gdf.to_parquet("irve.parquet")

# Check the data

In [6]:
%time
df = pd.read_parquet("/kaggle/working/irve.parquet")

CPU times: user 3 µs, sys: 0 ns, total: 3 µs
Wall time: 6.91 µs


In [7]:
df.shape, gdf.shape

((179863, 53), (179863, 53))

In [8]:
df.head()

,nom_amenageur,siren_amenageur,contact_amenageur,nom_operateur,contact_operateur,telephone_operateur,nom_enseigne,id_station_itinerance,id_station_local,nom_station,...,datagouv_organization_or_owner,created_at,consolidated_longitude,consolidated_latitude,consolidated_code_postal,consolidated_commune,consolidated_is_lon_lat_correct,consolidated_is_code_insee_verified,consolidated_is_code_insee_modified,geometry
0,ChargePoint,,info@chargepoint.com,ChargePoint,info@chargepoint.com,,ACU_Poste_De_Garde_Haguenau,ATHTBE1004017,ATHTBE1004017,ACU_Poste_De_Garde_Haguenau,...,eco-movement,2023-06-28 11:46:08.539000+00:00,7.762694,48.825613,,,False,False,False,b'\x01\x01\x00\x00\x00\xf4k\xeb\xa7\xff\x0c\x1...
1,ChargePoint,,info@chargepoint.com,ChargePoint,info@chargepoint.com,,ACU_Poste_De_Garde_Haguenau,ATHTBE1004018,ATHTBE1004018,ACU_Poste_De_Garde_Haguenau,...,eco-movement,2023-06-28 11:46:08.539000+00:00,7.762694,48.825613,,,False,False,False,b'\x01\x01\x00\x00\x00\xf4k\xeb\xa7\xff\x0c\x1...
2,ChargePoint,,info@chargepoint.com,ChargePoint,info@chargepoint.com,,ACU_Poste_De_Garde_Haguenau,ATHTBE1004019,ATHTBE1004019,ACU_Poste_De_Garde_Haguenau,...,eco-movement,2023-06-28 11:46:08.539000+00:00,7.762694,48.825613,,,False,False,False,b'\x01\x01\x00\x00\x00\xf4k\xeb\xa7\xff\x0c\x1...
3,ChargePoint,,info@chargepoint.com,ChargePoint,info@chargepoint.com,,ACU_Poste_De_Garde_Haguenau,ATHTBE1004020,ATHTBE1004020,ACU_Poste_De_Garde_Haguenau,...,eco-movement,2023-06-28 11:46:08.539000+00:00,7.762694,48.825613,,,False,False,False,b'\x01\x01\x00\x00\x00\xf4k\xeb\xa7\xff\x0c\x1...
4,ChargePoint,,info@chargepoint.com,ChargePoint,info@chargepoint.com,,HAG_P22_Slave_3,ATHTBE1004957,ATHTBE1004957,HAG_P22_Slave_3,...,eco-movement,2023-06-28 11:46:08.539000+00:00,7.761841,48.827040,,,False,False,False,b'\x01\x01\x00\x00\x00\xfc\x01\x0f\x0c \x0c\x1...


# Final remarks

We can load now the same data to the dataset, in the new parquet format. This will allow the user to load the data much faster, and with some of the filtering options.